In [1]:
import numpy as np

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [3]:
def sigmoid_derivative(x):
    return x * (1 - x)

In [4]:
def tanh(x):
    return np.tanh(x)

In [5]:
def tanh_derivative(x):
    return 1 - x ** 2

In [6]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_layers, hidden_neurons, output_size, activation_func):
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.hidden_neurons = hidden_neurons
        self.output_size = output_size
        self.activation_func = sigmoid if activation_func == 0 else tanh
        self.activation_derivative = sigmoid_derivative if activation_func == 0 else tanh_derivative
        
        self.weights = []
        self.biases = []
        
        prev_size = input_size
        for i in range(hidden_layers):
            self.weights.append(np.random.rand(prev_size, hidden_neurons[i]) - 0.5)
            self.biases.append(np.random.rand(hidden_neurons[i]) - 0.5)
            prev_size = hidden_neurons[i]
        
        self.weights.append(np.random.rand(prev_size, output_size) - 0.5)
        self.biases.append(np.random.rand(output_size) - 0.5)


    def forward(self, data):
        self.layer_outputs = []
        input_data = data
        
        for i in range(self.hidden_layers):
            value = np.dot(input_data, self.weights[i]) + self.biases[i]
            input_data = self.activation_func(value)
            self.layer_outputs.append(input_data)
        
        res = np.dot(input_data, self.weights[-1]) + self.biases[-1]
        output = self.activation_func(res)
        self.layer_outputs.append(output)
        return output


    def backward(self, data, expected, learning_rate):
        errors = [expected - self.layer_outputs[-1]]
        deltas = [errors[0] * self.activation_derivative(self.layer_outputs[-1])]

        for i in range(self.hidden_layers, 0, -1):
            error = np.dot(deltas[-1], self.weights[i].T)
            delta = error * self.activation_derivative(self.layer_outputs[i - 1])
            errors.append(error)
            deltas.append(delta)
        
        deltas.reverse()
        
        input_data = data
        for i in range(self.hidden_layers):
            self.weights[i] += np.dot(input_data.T, deltas[i]) * learning_rate
            self.biases[i] += np.sum(deltas[i], axis=0) * learning_rate
            input_data = self.layer_outputs[i]
        
        self.weights[-1] += np.dot(self.layer_outputs[-2].T, deltas[-1]) * learning_rate
        self.biases[-1] += np.sum(deltas[-1], axis=0) * learning_rate


    def train(self, data, expected, epochs, learning_rate):
        for _ in range(epochs):
            self.forward(data)
            self.backward(data, expected, learning_rate)

In [7]:
def boolean_function(name, X):
    if name == "AND":
        return np.array([[x[0] & x[1]] for x in X])
    elif name == "OR":
        return np.array([[x[0] | x[1]] for x in X])
    elif name == "XOR":
        return np.array([[x[0] ^ x[1]] for x in X])
    else:
        raise ValueError("Unsupported boolean function!")

In [8]:
def test_network(function_name, activation_func, hidden_layers, hidden_neurons):
    data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    expected = boolean_function(function_name, data)

    nn = NeuralNetwork(input_size=2, hidden_layers=hidden_layers, hidden_neurons=hidden_neurons, output_size=1, activation_func=activation_func)
    nn.train(data, expected, epochs=10000, learning_rate=0.1)

    print(f"{function_name}:")
    for x in data:
        output = nn.forward(x.reshape(1, -1))
        print(f"{tuple(x)} -> {output[0][0]:.4f}")

In [9]:
hidden_layers = 1
hidden_neurons = [4]

In [10]:
test_network("AND", 0, hidden_layers, hidden_neurons)

AND:
(np.int64(0), np.int64(0)) -> 0.0001
(np.int64(0), np.int64(1)) -> 0.0234
(np.int64(1), np.int64(0)) -> 0.0237
(np.int64(1), np.int64(1)) -> 0.9624


In [11]:
test_network("AND", 1, hidden_layers, hidden_neurons)

AND:
(np.int64(0), np.int64(0)) -> -0.0001
(np.int64(0), np.int64(1)) -> 0.0001
(np.int64(1), np.int64(0)) -> 0.0001
(np.int64(1), np.int64(1)) -> 0.9921


In [12]:
test_network("OR", 0, hidden_layers, hidden_neurons)

OR:
(np.int64(0), np.int64(0)) -> 0.0309
(np.int64(0), np.int64(1)) -> 0.9806
(np.int64(1), np.int64(0)) -> 0.9798
(np.int64(1), np.int64(1)) -> 0.9981


In [13]:
test_network("OR", 1, hidden_layers, hidden_neurons)

OR:
(np.int64(0), np.int64(0)) -> 0.0001
(np.int64(0), np.int64(1)) -> 0.9952
(np.int64(1), np.int64(0)) -> 0.9948
(np.int64(1), np.int64(1)) -> 0.9996


In [14]:
test_network("XOR", 0, hidden_layers, hidden_neurons)

XOR:
(np.int64(0), np.int64(0)) -> 0.0945
(np.int64(0), np.int64(1)) -> 0.9104
(np.int64(1), np.int64(0)) -> 0.9141
(np.int64(1), np.int64(1)) -> 0.0933


In [15]:
test_network("XOR", 1, hidden_layers, hidden_neurons)

XOR:
(np.int64(0), np.int64(0)) -> 0.0001
(np.int64(0), np.int64(1)) -> 0.9923
(np.int64(1), np.int64(0)) -> 0.9917
(np.int64(1), np.int64(1)) -> 0.0002
